# Data Processing for Movie Recomenditation System using Selenium

In [ ]:
!pip3 install selenium

## 1. Data Scraping and Storage

In [27]:
import pandas as pd
import time
from selenium.webdriver.common.by import By
from selenium import webdriver

url="https://www.imdb.com/search/title/?title_type=feature&release_date=2024-01-01,2024-12-31"
driver=webdriver.Chrome()
driver.get(url)
IBMD=driver.find_elements(By.CLASS_NAME,'ipc-metadata-list-summary-item')
time.sleep(10)
Name=[]
Rating=[]
Story=[]
for movie in IBMD:
    name=(movie.find_element(By.CLASS_NAME,'ipc-title__text')).text
    rating=(movie.find_element(By.CLASS_NAME,'ipc-rating-star--rating')).text
    story=(movie.find_element(By.CLASS_NAME,'ipc-html-content-inner-div')).text
    Name.append(name)
    Rating.append(rating)
    Story.append(story)
driver.quit()
df=pd.DataFrame({
    "Name":Name,
    "Rating":Rating,
    "Story":Story
})
df

,Name,Rating,Story
0,1. The Fall Guy,6.8,"A stuntman, fresh off an almost career-ending ..."
1,2. The Substance,7.2,A fading celebrity takes a black-market drug: ...
2,3. The Life of Chuck,7.3,"A life-affirming, genre-bending story about th..."
3,4. Abigail,6.5,After a group of criminals kidnap the ballerin...
4,5. The Ministry of Ungentlemanly Warfare,6.8,The British military recruits a small group of...
5,6. Relay,7.0,A broker of lucrative payoffs between corrupt ...
6,7. Dune: Part Two,8.4,Paul Atreides unites with the Fremen while on ...
7,8. Anora,7.4,A young stripper from Brooklyn meets and impul...
8,9. Jab Khuli Kitaab,9.4,Gopal and Anusuya's decades-long marriage face...
9,10. I'm Still Here,8.1,A woman married to a former politician during ...


## 2. Data Processing and Analysis

In [29]:
df.to_csv('C:\Avneesh\DSA\project\Project-5\project\data\IMBD_Data.csv')

<>:1: SyntaxWarning: invalid escape sequence '\A'
<>:1: SyntaxWarning: invalid escape sequence '\A'
C:\Users\avnee\AppData\Local\Temp\ipykernel_25588\1064067019.py:1: SyntaxWarning: invalid escape sequence '\A'
  df.to_csv('C:\Avneesh\DSA\project\Project-5\project\data\IMBD_Data.csv')


In [39]:
path=r"C:\Avneesh\DSA\project\Project-5\project\data\IMBD_Data.csv"
df=pd.read_csv(path)
df.shape

(50, 4)

In [45]:
df.head()

,Unnamed: 0,Name,Rating,Story
0,0,1. The Fall Guy,6.8,"A stuntman, fresh off an almost career-ending ..."
1,1,2. The Substance,7.2,A fading celebrity takes a black-market drug: ...
2,2,3. The Life of Chuck,7.3,"A life-affirming, genre-bending story about th..."
3,3,4. Abigail,6.5,After a group of criminals kidnap the ballerin...
4,4,5. The Ministry of Ungentlemanly Warfare,6.8,The British military recruits a small group of...


In [46]:
df.tail()

,Unnamed: 0,Name,Rating,Story
45,45,46. Trap,5.8,A father and his teen daughter attend a pop co...
46,46,47. I Was a Stranger,8.4,"Five strangers are pulled together, colliding ..."
47,47,48. Big World,7.5,"Liu Chunhe, a young man with cerebral palsy, o..."
48,48,49. IF,6.4,A young girl who goes through a difficult expe...
49,49,50. MaXXXine,6.2,"In 1980s Hollywood, adult film star and aspiri..."


In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  50 non-null     int64  
 1   Name        50 non-null     object 
 2   Rating      50 non-null     float64
 3   Story       50 non-null     object 
dtypes: float64(1), int64(1), object(2)
memory usage: 1.7+ KB


In [42]:
df.columns

Index(['Unnamed: 0', 'Name', 'Rating', 'Story'], dtype='object')

In [44]:
df.describe()

,Unnamed: 0,Rating
count,50.00000,50.000000
mean,24.50000,6.814000
std,14.57738,0.904188
min,0.00000,4.700000
25%,12.25000,6.400000
50%,24.50000,6.800000
75%,36.75000,7.300000
max,49.00000,9.400000


# NLP Data Processing

In [24]:
import pandas as pd
import nltk
import re
from nltk.corpus import stopwords

nltk.download('stopwords')

df = pd.read_csv("C:\Avneesh\DSA\project\Project-5\project\data\IMBD_Data.csv")

def clean_text(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z]', ' ', text)

    words = text.split()

    words = [w for w in words if w not in stopwords.words('english')]

    return " ".join(words)

df["clean_story"] = df["Story"].apply(clean_text)

<>:8: SyntaxWarning: invalid escape sequence '\A'
<>:8: SyntaxWarning: invalid escape sequence '\A'
C:\Users\avnee\AppData\Local\Temp\ipykernel_11860\2235966887.py:8: SyntaxWarning: invalid escape sequence '\A'
  df = pd.read_csv("C:\Avneesh\DSA\project\Project-5\project\data\IMBD_Data.csv")
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\avnee\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(df["clean_story"])
similarity_matrix = cosine_similarity(tfidf_matrix)

# Storyline Similarity

In [27]:
def recommend_movies(storyline):

    storyline = clean_text(storyline)

    vector = vectorizer.transform([storyline])

    similarity = cosine_similarity(vector, tfidf_matrix)

    scores = similarity[0]

    top_indices = scores.argsort()[-5:][::-1]

    return df.iloc[top_indices][["movie_name","storyline"]]